### 1. Configuración del directorio del proyecto
Crearemos la estructura de directorios solicitada: `data/`, `src/`, `README.md` y `requirements.txt`.

In [ ]:
import os

# Create directory structure
os.makedirs('data', exist_ok=True)
os.makedirs('src', exist_ok=True)

# Create requirements.txt
with open('requirements.txt', 'w') as f:
    f.write('torch\ntorchvision\nscikit-learn\nmatplotlib\npandas\n')

print("Directory structure and requirements.txt successfully created!")

### 2. PyTorch Configuration and Model Definition
We will detect the device (GPU if available, otherwise CPU) and define a simple Multilayer Perceptron (MLP) classification architecture using PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Simple MLP Architecture
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(SimpleMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.network(x)

### 3. Dataset Generation and Loader Setup
We generate a synthetic classification dataset, split it into train/validation sets, and wrap them in PyTorch DataLoaders.

In [ ]:
# Generate synthetic dataset
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch Tensors
train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### 4. Training and Evaluation Pipeline
We will define the training loop using the Adam optimizer and write an evaluation function to compute loss and accuracy.

In [ ]:
# Model instantiation
input_dim = 20
hidden_dim = 64
num_classes = 2
learning_rate = 0.005
epochs = 15

model = SimpleMLP(input_dim, hidden_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss / total, correct / total

# Training Loop
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1:02d}/{epochs} -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

### 5. Training History Plot
Let's plot the training and validation loss/accuracy curves.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss Plot
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_title('Cross Entropy Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Accuracy Plot
ax2.plot(train_accs, label='Train Acc')
ax2.plot(val_accs, label='Val Acc')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

### 6. Generate README.md Documentation

In [ ]:
readme_content = f"""# Pipeline Base Project

## Setup and Environment
- **PyTorch Version**: {torch.__version__}
- **Device Utilized**: {device}

## Hyperparameters
- **Model Architecture**: MLP (20 -> 64 -> 2 with ReLU activation)
- **Optimizer**: Adam
- **Learning Rate**: {learning_rate}
- **Batch Size**: 32
- **Epochs**: {epochs}

## Results and Interpretations
During training, we expect to see the loss values for both training and validation sets steadily decrease. If the train loss decreases but the validation loss begins to rise, that would signal overfitting.
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print("README.md successfully generated and documented.")